In [1]:
# Notebook 02_4 — Baseline classifiers and evaluation
# This notebook:
# 1) Loads preprocessed features and splits from Notebook 3
# 2) Trains baseline classifiers on different feature combinations:
#    - Patterns-only (sparse binary features from mined itemsets)
#    - Embeddings-only (dense multilingual sentence embeddings)
#    - Fused features (patterns + embeddings combined)
# 3) Evaluates each variant with accuracy, precision, recall, F1, confusion matrix
# 4) Saves trained models and generates comparison reports

import os
import json
import random
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# IO paths
PROCESSED_DIR = "data/processed"
RESULTS_DIR = "results"
MODELS_DIR = "models"
REPORTS_DIR = "docs/reports"

# Create directories
for d in [RESULTS_DIR, MODELS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Model configuration
# We'll test multiple algorithms to see which works best for each feature type
MODELS_TO_TEST = {
    "LogisticRegression": {"C": 1.0, "max_iter": 1000, "random_state": SEED},
    "RandomForest": {"n_estimators": 100, "max_depth": 10, "random_state": SEED},
    "SVM": {"C": 1.0, "kernel": "linear", "random_state": SEED},
}

TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
print("Config ready. Models to test:", list(MODELS_TO_TEST.keys()))

Config ready. Models to test: ['LogisticRegression', 'RandomForest', 'SVM']


In [2]:
from scipy import sparse
import joblib

# Check required files exist
required_files = [
    "feature_info.json", "y_train.npy", "y_val.npy", "y_test.npy",
    "X_patterns_train.npz", "X_embed_train.npy", "X_fused_std_train.npz"
]

missing_files = []
for f in required_files:
    if not os.path.exists(os.path.join(PROCESSED_DIR, f)):
        missing_files.append(f)

if missing_files:
    raise FileNotFoundError(f"Missing files from Notebook 3: {missing_files}")

# Load feature info and label mappings
with open(os.path.join(PROCESSED_DIR, "feature_info.json"), "r") as f:
    feature_info = json.load(f)

label_to_id = feature_info["label_to_id"]
id_to_label = feature_info["id_to_label"]
pattern_vocab = feature_info["pattern_vocab"]

print("Feature info loaded:")
print(f"- Classes: {len(label_to_id)}")
print(f"- Pattern vocab size: {len(pattern_vocab)}")
print(f"- Embedding model: {feature_info['embed_model']}")
print(f"- Embedding dim: {feature_info['embed_dim']}")

Feature info loaded:
- Classes: 3
- Pattern vocab size: 3
- Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Embedding dim: 384


In [3]:
# Load labels
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"))
y_val = np.load(os.path.join(PROCESSED_DIR, "y_val.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

# Load pattern features (sparse)
X_pat_train = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_patterns_train.npz"))
X_pat_val = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_patterns_val.npz"))
X_pat_test = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_patterns_test.npz"))

# Load embeddings (dense, standardized version for better linear model performance)
X_emb_train = np.load(os.path.join(PROCESSED_DIR, "X_embed_std_train.npy"))
X_emb_val = np.load(os.path.join(PROCESSED_DIR, "X_embed_std_val.npy"))
X_emb_test = np.load(os.path.join(PROCESSED_DIR, "X_embed_std_test.npy"))

# Load fused features (patterns + standardized embeddings)
X_fused_train = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_fused_std_train.npz"))
X_fused_val = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_fused_std_val.npz"))
X_fused_test = sparse.load_npz(os.path.join(PROCESSED_DIR, "X_fused_std_test.npz"))

print("Data loaded:")
print(f"- Train: {len(y_train)} samples")
print(f"- Val: {len(y_val)} samples")
print(f"- Test: {len(y_test)} samples")
print(f"- Pattern features: {X_pat_train.shape[1]}")
print(f"- Embedding features: {X_emb_train.shape[1]}")
print(f"- Fused features: {X_fused_train.shape[1]}")

Data loaded:
- Train: 1 samples
- Val: 1 samples
- Test: 1 samples
- Pattern features: 3
- Embedding features: 384
- Fused features: 387


In [4]:
# Define different feature combinations to evaluate
# Each combination has a name, train/val/test matrices, and description
feature_combinations = {
    "patterns_only": {
        "description": "Sparse binary features from mined adjective-noun patterns and OSM hints",
        "X_train": X_pat_train,
        "X_val": X_pat_val,
        "X_test": X_pat_test,
        "feature_type": "sparse"
    },
    "embeddings_only": {
        "description": "Dense multilingual sentence embeddings (standardized)",
        "X_train": X_emb_train,
        "X_val": X_emb_val,
        "X_test": X_emb_test,
        "feature_type": "dense"
    },
    "fused": {
        "description": "Combined patterns + embeddings (sparse + dense fused)",
        "X_train": X_fused_train,
        "X_val": X_fused_val,
        "X_test": X_fused_test,
        "feature_type": "sparse"  # fused matrix is sparse
    }
}

print("Feature combinations to test:")
for name, info in feature_combinations.items():
    print(f"- {name}: {info['X_train'].shape} ({info['feature_type']})")

Feature combinations to test:
- patterns_only: (1, 3) (sparse)
- embeddings_only: (1, 384) (dense)
- fused: (1, 387) (sparse)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import joblib

# Model factory function
def create_model(model_name, params):
    """Create a sklearn model instance with given parameters"""
    if model_name == "LogisticRegression":
        return LogisticRegression(**params)
    elif model_name == "RandomForest":
        return RandomForestClassifier(**params)
    elif model_name == "SVM":
        return SVC(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")

print("Sklearn imports ready. Model factory defined.")

Sklearn imports ready. Model factory defined.


In [6]:
def train_and_evaluate(model, X_train, y_train, X_val, y_val, X_test, y_test,
                      model_name, feature_name, id_to_label):
    """
    Train a model and evaluate on validation and test sets.
    Returns a dictionary with all metrics and predictions.
    """
    print(f"Training {model_name} on {feature_name}...")

    # Train the model
    model.fit(X_train, y_train)

    # Predictions on all sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    # Calculate metrics for each set
    def calc_metrics(y_true, y_pred, set_name):
        return {
            f"{set_name}_accuracy": accuracy_score(y_true, y_pred),
            f"{set_name}_precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
            f"{set_name}_recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
            f"{set_name}_f1": f1_score(y_true, y_pred, average='weighted', zero_division=0)
        }

    results = {}
    results.update(calc_metrics(y_train, y_train_pred, "train"))
    results.update(calc_metrics(y_val, y_val_pred, "val"))
    results.update(calc_metrics(y_test, y_test_pred, "test"))

    # Store predictions and model info
    results.update({
        "model_name": model_name,
        "feature_name": feature_name,
        "y_val_pred": y_val_pred,
        "y_test_pred": y_test_pred,
        "trained_model": model
    })

    # Generate classification report for test set
    class_names = [id_to_label[str(i)] for i in sorted([int(k) for k in id_to_label.keys()])]
    results["test_classification_report"] = classification_report(
        y_test, y_test_pred, target_names=class_names, zero_division=0
    )

    print(f"  Test accuracy: {results['test_accuracy']:.3f}")
    print(f"  Test F1: {results['test_f1']:.3f}")

    return results

print("Training and evaluation function defined.")

Training and evaluation function defined.


In [7]:
# Store all results for comparison
all_results = []
trained_models = {}

# Loop through each feature combination
for feature_name, feature_data in feature_combinations.items():
    print(f"\n=== Testing feature combination: {feature_name} ===")
    print(f"Description: {feature_data['description']}")

    X_train = feature_data["X_train"]
    X_val = feature_data["X_val"]
    X_test = feature_data["X_test"]

    # Loop through each model type
    for model_name, model_params in MODELS_TO_TEST.items():
        try:
            # Create and train model
            model = create_model(model_name, model_params)

            # Train and evaluate
            results = train_and_evaluate(
                model, X_train, y_train, X_val, y_val, X_test, y_test,
                model_name, feature_name, id_to_label
            )

            # Store results
            all_results.append(results)

            # Save trained model
            model_key = f"{feature_name}_{model_name}"
            trained_models[model_key] = results["trained_model"]

            # Save model to disk
            model_path = os.path.join(MODELS_DIR, f"{model_key}.joblib")
            joblib.dump(results["trained_model"], model_path)

        except Exception as e:
            print(f"  ERROR training {model_name} on {feature_name}: {e}")
            continue

print(f"\nCompleted training {len(all_results)} model-feature combinations.")


=== Testing feature combination: patterns_only ===
Description: Sparse binary features from mined adjective-noun patterns and OSM hints
Training LogisticRegression on patterns_only...
  ERROR training LogisticRegression on patterns_only: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)
Training RandomForest on patterns_only...
  ERROR training RandomForest on patterns_only: Number of classes, 2, does not match size of target_names, 3. Try specifying the labels parameter
Training SVM on patterns_only...
  ERROR training SVM on patterns_only: The number of classes has to be greater than one; got 1 class

=== Testing feature combination: embeddings_only ===
Description: Dense multilingual sentence embeddings (standardized)
Training LogisticRegression on embeddings_only...
  ERROR training LogisticRegression on embeddings_only: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np